# ChuckleNet: Scale to 1000+ Videos
## Pipeline: Extract → Label → Train

**Data already on Drive:** `chuckle_net/` (620 audio, 628 VTT)
**Runtime**: ~3-4 hours on Colab (CPU prosody + GPU WavLM)
**Output**: Fusion model on 500-1000+ videos

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')

!pip install -q librosa numpy pandas scikit-learn torch transformers

import numpy as np
import glob

BASE = 'BASE_PATH_1000'
AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'

audio_files = sorted(glob.glob(f'{AUDIO_DIR}/*.m4a'))
vtt_files = sorted(glob.glob(f'{VTT_DIR}/*.vtt'))
print(f'Audio: {len(audio_files)} files')
print(f'VTT: {len(vtt_files)} files')
print(f'Audio dir: {AUDIO_DIR}')
print(f'VTT dir: {VTT_DIR}')

In [ ]:
# === STEP 1: Extract Prosody (CPU FAST) ===
import librosa
import numpy as np
from tqdm import tqdm

def extract_prosody(audio_path, sr=22050):
    """Extract 23-dim prosody features."""
    try:
        y, sr = librosa.load(audio_path, sr=sr)
        if len(y) < sr: return None
        
        # Features
        rms = librosa.feature.rms(y=y, hop_length=512)[0]
        f0, voiced, prob = librosa.pyin(y, fmin=50, fmax=500, sr=sr, hop_length=512)
        f0 = np.nan_to_num(f0, nan=0)
        voiced_f0 = f0[voiced] if voiced.any() else np.array([0])
        
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=512)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=512)[0]
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=512)[0]
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=512)
        
        return [
            np.mean(rms), np.std(rms), np.max(rms),
            np.mean(zcr), np.std(zcr),
            np.mean(sc), np.std(sc),
            np.mean(sb), np.std(sb),
            np.mean(rolloff), np.std(rolloff),
            np.mean(mfcc[0]), np.std(mfcc[0]),
            np.mean(mfcc[1]), np.mean(mfcc[2]), np.mean(mfcc[3]),
            np.mean(mfcc[4]), np.mean(mfcc[5]), np.mean(mfcc[6]),
            np.mean(mfcc[7]), np.mean(mfcc[8]), np.mean(mfcc[9]),
            np.mean(mfcc[10]),
            np.mean(voiced_f0) if len(voiced_f0)>0 else 0,
            np.std(voiced_f0) if len(voiced_f0)>1 else 0,
        ]
    except:
        return None

# Extract from all audio
all_features, all_labels, all_vids = [], [], []

for af in tqdm(audio_files):
    vid = os.path.basename(af).replace('.m4a', '')
    feat = extract_prosody(af)
    if feat is None: continue
    
    # Check VTT for [laughter]
    vtt_patterns = glob.glob(f'{VTT_DIR}/{vid}.*.vtt')
    has_laugh = False
    if vtt_patterns:
        try:
            with open(vtt_patterns[0]) as f:
                content = f.read().lower()
                has_laugh = '[laughter]' in content
        except: pass
    
    all_features.append(feat)
    all_labels.append(1 if has_laugh else 0)
    all_vids.append(vid)

X = np.array(all_features)
y = np.array(all_labels)
vids = np.array(all_vids)

print(f'\nExtracted: {len(X)} samples')
print(f'Positive (has [laughter]): {y.sum()} ({100*y.mean():.1f}%)')

# Save
np.savez_compressed(f'{BASE}/prosody_features.npz', features=X, labels=y, vids=vids)
print(f'Saved: {BASE}/prosody_features.npz')

In [ ]:
# === STEP 2: Extract WavLM (GPU) ===
import torch
from transformers import Wav2Vec2Model
import librosa
import numpy as np
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading WavLM...')
wavlm = Wav2Vec2Model.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()

def extract_wavlm(audio_path, sr=16000):
    try:
        y, _ = librosa.load(audio_path, sr=sr)
        if len(y) < sr: return None
        with torch.no_grad():
            inputs = torch.FloatTensor(y).unsqueeze(0).to(device)
            out = wavlm(inputs)
            emb = out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return emb
    except:
        return None

all_embs, valid_vids = [], []

for af in tqdm(audio_files, desc='WavLM'):
    vid = os.path.basename(af).replace('.m4a', '')
    emb = extract_wavlm(af)
    if emb is not None:
        all_embs.append(emb)
        valid_vids.append(vid)

X_wavlm = np.array(all_embs)
print(f'\nWavLM: {X_wavlm.shape}')

np.savez_compressed(f'{BASE}/wavlm_embeddings.npz', embeddings=X_wavlm, vids=np.array(valid_vids))
print(f'Saved: {BASE}/wavlm_embeddings.npz')

In [ ]:
# === STEP 3: Train Fusion Model ===
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

# Load
prosody_data = np.load(f'{BASE}/prosody_features.npz')
wavlm_data = np.load(f'{BASE}/wavlm_embeddings.npz')

X_p = prosody_data['features']
y_p = prosody_data['labels']
prosody_vids = prosody_data['vids']

wavlm_vids = wavlm_data['vids']
X_w = wavlm_data['embeddings']

# Match
wavlm_map = {str(v): X_w[i] for i, v in enumerate(wavlm_vids)}
matched_w, matched_p, matched_y, matched_v = [], [], [], []
for i, (pv, py) in enumerate(zip(prosody_vids, y_p)):
    if str(pv) in wavlm_map:
        matched_w.append(wavlm_map[str(pv)])
        matched_p.append(X_p[i])
        matched_y.append(py)
        matched_v.append(str(pv))

X_wm = np.array(matched_w)
X_pm = np.array(matched_p)
ym = np.array(matched_y)
vm = np.array(matched_v)

print(f'Matched: {len(ym)} samples, {len(set(vm))} videos')
print(f'Positive: {ym.sum()} ({100*ym.mean():.1f}%)')

# Standardize
scaler_w = StandardScaler().fit(X_wm)
scaler_p = StandardScaler().fit(X_pm)
X_ws = scaler_w.transform(X_wm)
X_ps = scaler_p.transform(X_pm)
X_f = np.concatenate([X_ws, X_ps], axis=1)

# Fusion MLP
class FusionMLP(nn.Module):
    def __init__(self, dim=791, hidden=[512, 256, 64]):
        super().__init__()
        self.bn0 = nn.BatchNorm1d(dim)
        layers = []
        prev = dim
        for h in hidden:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(self.bn0(x)).squeeze(-1)

# 5-fold video CV
gkf = GroupKFold(n_splits=5)
f1s = []

for fold, (tr, te) in enumerate(gkf.split(X_f, ym, vm)):
    X_tr, y_tr = torch.FloatTensor(X_f[tr]), torch.FloatTensor(ym[tr])
    X_te = torch.FloatTensor(X_f[te])
    
    model = FusionMLP(dim=X_f.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    pos_w = torch.tensor((1-y_tr.mean())/max(0.01, y_tr.mean()) ).to(device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    
    for epoch in range(100):
        model.train()
        opt.zero_grad()
        out = model(X_tr.to(device))
        loss = loss_fn(out, y_tr.to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
    
    model.eval()
    with torch.no_grad():
        pred = (torch.sigmoid(model(X_te.to(device))) > 0.5).cpu().numpy().astype(int)
    f1 = f1_score(ym[te], pred, zero_division=0)
    f1s.append(f1)
    print(f'Fold {fold+1}: F1={f1:.4f} (pos={ym[te].mean():.1%})')

print(f'\nMean F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
torch.save(model.state_dict(), f'{BASE}/fusion_model_1000.pt')
print(f'Saved: {BASE}/fusion_model_1000.pt')